E01: Tune the hyperparameters of the training to beat my best validation loss of 2.2

In [1]:
import torch
import torch.nn.functional as F
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [3]:
def build_dataset(words, block_size):
    X, y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:] + [ix] 

    X = torch.tensor(X)
    y = torch.tensor(y)

    return X, y

In [4]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
block_size = 3

Xtr, ytr = build_dataset(words[:n1], block_size)
Xdev, ydev = build_dataset(words[n1:n2], block_size)
Xte, yte = build_dataset(words[n2:], block_size)

In [5]:
class MLP:
    def __init__(self, embed_size: int, hidden_layer_size: int, block_size: int,seed: int = -1):
        # Seed
        self.seed = seed if seed != -1 else torch.randint(low=1, high=50000, size=()).item()
        self.g = torch.Generator().manual_seed(self.seed)

        # Object info
        self.embed_size = embed_size
        self.hidden_layer_size = hidden_layer_size
        self.block_size = block_size

        # Randomly initialize parameters
        self.C = torch.randn((27, embed_size), requires_grad=True, generator=self.g)
        self.W1 = torch.randn((embed_size * block_size, hidden_layer_size), requires_grad=True, generator=self.g)
        self.b1 = torch.randn(hidden_layer_size, requires_grad=True, generator=self.g)
        self.W2 = torch.randn((hidden_layer_size, 27), requires_grad=True, generator=self.g)
        self.b2 = torch.randn(27, requires_grad=True, generator=self.g)
        self.parameters = [self.C, self.W1, self.b1, self.W2, self.b2]

    def train(self, X: torch.Tensor, y: torch.Tensor, num_iterations: int = 10000, batch_size: int = 32, learn_rate: list[float] = [0.1, 0.02, 0.005]):
        for lr in learn_rate:
            for _ in range(num_iterations):
                #batch construct
                ix = torch.randint(0, X.shape[0], (batch_size, ))
            
                # forward pass
                emb = self.C[X[ix]]
                h = torch.tanh(emb.view(-1, self.embed_size * self.block_size) @ self.W1 + self.b1)
                logits = h @ self.W2 + self.b2
                loss = F.cross_entropy(logits, y[ix])

                #backward pass
                for p in self.parameters:
                    p.grad = None
                loss.backward()

                #update
                for p in self.parameters:
                    p.data += -lr * p.grad

        print("Train loss: ", self.eval(X, y))

    def eval(self, X: torch.Tensor, y: torch.Tensor):
        emb = self.C[X]
        h = torch.tanh(emb.view(-1, self.embed_size * self.block_size) @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        loss = F.cross_entropy(logits, y)
        return loss.item()

    def num_params(self):
        return sum(p for p in self.parameters)

Here I emulate sorta what Karpathy did hyperparamter-wise to establish the baseline to beat with my different seeding.

In [6]:
base_mlp = MLP(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
base_mlp.train(Xtr, ytr, num_iterations=30000, learn_rate=[0.1, 0.01])

Train loss:  2.211392641067505


In [7]:
base_mlp.eval(Xdev, ydev)

2.2324767112731934

In [8]:
base_test_loss = base_mlp.eval(Xte, yte)
base_test_loss

2.2333428859710693

Now I'll try to improve on this with a few experiments (E01). Here are the different hyperparamter configurations I'll test(any unwritten variables will be the same as the base model):
- hidden_layer_size=100, 200, 300, 400
- embed_size=2,5,10,15
- num_iterations=30000, 50000, 100000
- learn_rate=[0.1, 0.01], [0.1, 0.02, 0.005], [0.15, 0.03, 0.005], [0.05, 0.01, 0.002]

Then I'll combine the ones that perform best of these and see if the combination does any better.

In [9]:
for hls in [100, 200, 300, 400]:
    hidden_test = MLP(embed_size=10, hidden_layer_size=hls, block_size=block_size, seed=42)
    hidden_test.train(Xtr, ytr, num_iterations=30000, learn_rate=[0.1, 0.01])
    print("Loss for hidden_layer_size=", hls, ": ", hidden_test.eval(Xdev, ydev))

Train loss:  2.2233293056488037
Loss for hidden_layer_size= 100 :  2.236912727355957
Train loss:  2.2221388816833496
Loss for hidden_layer_size= 200 :  2.235499858856201
Train loss:  2.212994337081909
Loss for hidden_layer_size= 300 :  2.2493348121643066
Train loss:  2.2178361415863037
Loss for hidden_layer_size= 400 :  2.2586352825164795


In [10]:
for es in [2, 5, 10, 15]:
    embed_test = MLP(embed_size=es, hidden_layer_size=200, block_size=block_size, seed=42)
    embed_test.train(Xtr, ytr, num_iterations=30000, learn_rate=[0.1, 0.01])
    print("Loss for embed_size=", es, ": ", embed_test.eval(Xdev, ydev))

Train loss:  2.3152740001678467
Loss for embed_size= 2 :  2.3157684803009033
Train loss:  2.2499876022338867
Loss for embed_size= 5 :  2.2524006366729736
Train loss:  2.204951763153076
Loss for embed_size= 10 :  2.2200424671173096
Train loss:  2.1857519149780273
Loss for embed_size= 15 :  2.2318718433380127


In [11]:
for ni in [30000, 50000, 100000]:
    num_its_test = MLP(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
    num_its_test.train(Xtr, ytr, num_iterations=ni, learn_rate=[0.1, 0.01])
    print("Loss for num_iterations=", ni, ": ", num_its_test.eval(Xdev, ydev))

Train loss:  2.2124788761138916
Loss for num_iterations= 30000 :  2.227076768875122
Train loss:  2.162522315979004
Loss for num_iterations= 50000 :  2.190709114074707
Train loss:  2.115818977355957
Loss for num_iterations= 100000 :  2.155104160308838


In [12]:
for lr in [[0.1, 0.01], [0.1, 0.02, 0.005], [0.15, 0.03, 0.005], [0.05, 0.01, 0.002]]:
    lr_test = MLP(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
    lr_test.train(Xtr, ytr, num_iterations=30000, learn_rate=lr)
    print("Loss for learn_rate=", lr, ": ", lr_test.eval(Xdev, ydev))

Train loss:  2.20521879196167
Loss for learn_rate= [0.1, 0.01] :  2.2196011543273926
Train loss:  2.183690309524536
Loss for learn_rate= [0.1, 0.02, 0.005] :  2.197977304458618
Train loss:  2.147491931915283
Loss for learn_rate= [0.15, 0.03, 0.005] :  2.1782097816467285
Train loss:  2.230149269104004
Loss for learn_rate= [0.05, 0.01, 0.002] :  2.2418956756591797


Combine the best parameters(gonna do two hidden layer sizes since It seems more iterations and finer-tuned learning rate helps with more parameters):
- hidden_layer_size=100, 200
- embed_size=10
- num_its=100000
- lr=[0.15, 0.03, 0.005]

In [13]:
for hls in [100, 200]:
    combo_test = MLP(embed_size=10, hidden_layer_size=hls, block_size=block_size, seed=42)
    combo_test.train(Xtr, ytr, num_iterations=100000, learn_rate=[0.15, 0.03, 0.005])
    print("Loss for combo test(hls=", hls, "): ", combo_test.eval(Xdev, ydev))

Train loss:  2.117540121078491
Loss for combo test(hls= 100 ):  2.149401903152466
Train loss:  2.0787622928619385
Loss for combo test(hls= 200 ):  2.1315646171569824


The best loss I achieved was the model with parameters: 
- embed_size=10
- hidden_layer_size=200
- num_iterations=100000
- learn_rate=[0.15, 0.03, 0.005]

In [15]:
best_model = MLP(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
best_model.train(Xtr, ytr, num_iterations=100000, learn_rate=[0.15, 0.03, 0.005])
print("Dev loss for best model:", combo_test.eval(Xdev, ydev))
print("Test loss for best model:", combo_test.eval(Xte, yte))

Train loss:  2.094020128250122
Dev loss for best model: 2.1315646171569824
Test loss for best model: 2.128044605255127


In [16]:
base_test_loss

2.2333428859710693

I was able to get my model to get a better test loss by a pretty decent amount with some hyperparameter tuning.

E02: I was not careful with the intialization of the network in this video. (1) What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we achieve? (2) Can you tune the initialization to get a starting loss that is much more similar to (1)?

(1) 

$$L=-\sum_{i=1}^{26}0\cdot\log\left(\frac{1}{27}\right)-1\cdot\log\left(\frac{1}{27}\right)=3.296$$

If output probabilities were uniform, loss would be 3.296. We saw a loss of 17-19 on initialization in the video, so we can definitely improve initialization here.

(2) below:

In [17]:
class MLP_better_init:
    def __init__(self, embed_size: int, hidden_layer_size: int, block_size: int,seed: int = -1):
        # Seed
        self.seed = seed if seed != -1 else torch.randint(low=1, high=50000, size=()).item()
        self.g = torch.Generator().manual_seed(self.seed)

        # Object info
        self.embed_size = embed_size
        self.hidden_layer_size = hidden_layer_size
        self.block_size = block_size

        # Randomly initialize parameters
        self.C = torch.randn((27, embed_size), generator=self.g)
        self.W1 = torch.randn((embed_size * block_size, hidden_layer_size), generator=self.g)
        self.b1 = torch.randn(hidden_layer_size, generator=self.g)
        self.W2 = torch.randn((hidden_layer_size, 27), generator=self.g) / 100
        self.b2 = torch.zeros(27)
        self.parameters = [self.C, self.W1, self.b1, self.W2, self.b2]
        for p in self.parameters:
            p.requires_grad = True

    def train(self, X: torch.Tensor, y: torch.Tensor, num_iterations: int = 10000, batch_size: int = 32, learn_rate: list[float] = [0.1, 0.02, 0.005]):
        init = True
        for lr in learn_rate:
            for _ in range(num_iterations):
                #batch construct
                ix = torch.randint(0, X.shape[0], (batch_size, ))
            
                # forward pass
                emb = self.C[X[ix]]
                h = torch.tanh(emb.view(-1, self.embed_size * self.block_size) @ self.W1 + self.b1)
                logits = h @ self.W2 + self.b2
                loss = F.cross_entropy(logits, y[ix])
                if init:
                    print("Initial loss: ", loss)
                    init = False

                #backward pass
                for p in self.parameters:
                    p.grad = None
                loss.backward()

                #update
                for p in self.parameters:
                    p.data += -lr * p.grad

        print("Train loss: ", self.eval(X, y))

    def eval(self, X: torch.tensor, y: torch.tensor):
        emb = self.C[X]
        h = torch.tanh(emb.view(-1, self.embed_size * self.block_size) @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        loss = F.cross_entropy(logits, y)
        return loss.item()

    def num_params(self):
        return sum(p for p in self.parameters)

In [18]:
best_model = MLP_better_init(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
best_model.train(Xtr, ytr, num_iterations=100000, learn_rate=[0.15, 0.03, 0.005])
print("Dev loss for best model:", best_model.eval(Xdev, ydev))
print("Test loss for best model:", best_model.eval(Xte, yte))

Initial loss:  tensor(3.3191, grad_fn=<NllLossBackward0>)
Train loss:  2.0442163944244385
Dev loss for best model: 2.1171255111694336
Test loss for best model: 2.1162331104278564


E03: Read the Bengio et al 2003 paper, implement and try any idea from the paper. Did it work?

I'm implementing the direct, linear connections from the embeddings to the softmax layer. It slightly lowers the loss compared with the model without it above.

In [19]:
class MLP_with_directs:
    def __init__(self, embed_size: int, hidden_layer_size: int, block_size: int,seed: int = -1):
        # Seed
        self.seed = seed if seed != -1 else torch.randint(low=1, high=50000, size=()).item()
        self.g = torch.Generator().manual_seed(self.seed)

        # Object info
        self.embed_size = embed_size
        self.hidden_layer_size = hidden_layer_size
        self.block_size = block_size

        # Randomly initialize parameters
        self.C = torch.randn((27, embed_size), generator=self.g)
        self.W1 = torch.randn((embed_size * block_size, hidden_layer_size), generator=self.g)
        self.b1 = torch.randn(hidden_layer_size, generator=self.g)
        self.W2 = torch.randn((hidden_layer_size, 27), generator=self.g) / 100
        self.b2 = torch.zeros(27)
        self.directs = torch.randn((embed_size * block_size, 27), generator=self.g) / 100
        self.parameters = [self.C, self.W1, self.b1, self.W2, self.b2, self.directs]
        for p in self.parameters:
            p.requires_grad = True

    def train(self, X: torch.tensor, y: torch.tensor, num_iterations: int = 10000, batch_size: int = 32, learn_rate: list[float] = [0.1, 0.02, 0.005]):
        init = True
        for lr in learn_rate:
            for _ in range(num_iterations):
                #batch construct
                ix = torch.randint(0, X.shape[0], (batch_size, ))
            
                # forward pass
                emb = self.C[X[ix]]
                emb_concat = emb.view(-1, self.embed_size * self.block_size)
                h = torch.tanh(emb_concat @ self.W1 + self.b1)
                logits = h @ self.W2 + emb_concat @ self.directs + self.b2
                loss = F.cross_entropy(logits, y[ix])
                if init:
                    print("Initial loss: ", loss)
                    init = False

                #backward pass
                for p in self.parameters:
                    p.grad = None
                loss.backward()

                #update
                for p in self.parameters:
                    p.data += -lr * p.grad

        print("Train loss: ", self.eval(X, y))

    def eval(self, X: torch.tensor, y: torch.tensor):
        emb = self.C[X]
        emb_concat = emb.view(-1, self.embed_size * self.block_size)
        h = torch.tanh(emb_concat @ self.W1 + self.b1)
        logits = h @ self.W2 + emb_concat @ self.directs + self.b2
        loss = F.cross_entropy(logits, y)
        return loss.item()

    def num_params(self):
        return sum(p.nelement() for p in self.parameters)

In [20]:
best_model = MLP_with_directs(embed_size=10, hidden_layer_size=200, block_size=block_size, seed=42)
best_model.train(Xtr, ytr, num_iterations=100000, learn_rate=[0.15, 0.03, 0.005])
print("Dev loss for best model:", best_model.eval(Xdev, ydev))
print("Test loss for best model:", best_model.eval(Xte, yte))

Initial loss:  tensor(3.2675, grad_fn=<NllLossBackward0>)
Train loss:  2.044093370437622
Dev loss for best model: 2.1112260818481445
Test loss for best model: 2.116203546524048


In [ ]:
best_model.num_params()

12707